<a href="https://colab.research.google.com/github/KarolinaBaumert/Python2/blob/automatic_plate_number_recognition/Zaawansowane_programowanie_projekt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Trenowanie**


In [ ]:
import xml.etree.ElementTree as ET
import os
import random
import shutil

In [ ]:
# Ścieżki do danych
xml_path = "/content/drive/MyDrive/Colab_Notebooks/dane/poland-vehicle-license-plate-dataset/annotations.xml"
images_path = "/content/drive/MyDrive/Colab_Notebooks/dane/poland-vehicle-license-plate-dataset/photos"
base_path = "/content/drive/MyDrive/Colab_Notebooks/dane/datasets/processed_data"

In [ ]:
# Ścieżki do folderów YOLO
images_train = os.path.join(base_path, "images/train")
images_val = os.path.join(base_path, "images/val")
labels_train = os.path.join(base_path, "labels/train")
labels_val = os.path.join(base_path, "labels/val")


In [ ]:
# Tworzenie folderów
os.makedirs(images_train, exist_ok=True)
os.makedirs(images_val, exist_ok=True)
os.makedirs(labels_train, exist_ok=True)
os.makedirs(labels_val, exist_ok=True)


In [ ]:
# Funkcja do konwersji adnotacji do formatu YOLO
def convert_to_yolo_format(annotation, image_width, image_height):
    x_min = int(float(annotation['xtl']))
    y_min = int(float(annotation['ytl']))
    x_max = int(float(annotation['xbr']))
    y_max = int(float(annotation['ybr']))
    x_center = (x_min + x_max) / 2 / image_width
    y_center = (y_min + y_max) / 2 / image_height
    width = (x_max - x_min) / image_width
    height = (y_max - y_min) / image_height
    return x_center, y_center, width, height

In [ ]:
# Załaduj XML
tree = ET.parse(xml_path)
root = tree.getroot()


In [ ]:
# Tymczasowy folder na pliki .txt
tmp_labels_path = os.path.join(base_path, "tmp_labels")
os.makedirs(tmp_labels_path, exist_ok=True)

In [ ]:
# Tworzenie plików .txt z adnotacjami
all_txt_files = []
for image in root.findall('image'):
    image_name = image.get('name')
    width = int(image.get('width'))
    height = int(image.get('height'))
    txt_filename = os.path.splitext(image_name)[0] + ".txt"
    txt_filepath = os.path.join(tmp_labels_path, txt_filename)
    with open(txt_filepath, 'w') as f:
        for box in image.findall('box'):
            if box.get('label') == 'plate':
                x_center, y_center, w, h = convert_to_yolo_format(box.attrib, width, height)
                f.write(f"0 {x_center} {y_center} {w} {h}\n")
    all_txt_files.append(txt_filename)

In [ ]:
# Losowy podział na trening i walidację
random.shuffle(all_txt_files)
split_index = int(0.7 * len(all_txt_files))
train_files = all_txt_files[:split_index]
val_files = all_txt_files[split_index:]

In [ ]:
# Przenoszenie danych do struktur YOLO
for file in train_files:
    base_name = os.path.splitext(file)[0]
    shutil.copy(os.path.join(tmp_labels_path, file), os.path.join(labels_train, file))
    shutil.copy(os.path.join(images_path, base_name + ".jpg"), os.path.join(images_train, base_name + ".jpg"))

for file in val_files:
    base_name = os.path.splitext(file)[0]
    shutil.copy(os.path.join(tmp_labels_path, file), os.path.join(labels_val, file))
    shutil.copy(os.path.join(images_path, base_name + ".jpg"), os.path.join(images_val, base_name + ".jpg"))

# Usunięcie tymczasowego folderu
shutil.rmtree(tmp_labels_path)

In [ ]:
# Tworzenie pliku data.yaml
with open(os.path.join(base_path, "data.yaml"), "w") as f:
    f.write(
        f"path: {base_path}\n"
        "train: images/train\n"
        "val: images/val\n"
        "names:\n"
        "  0: plate\n"
    )


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 122.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 91.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [ ]:
import ultralytics
ultralytics.checks()

Ultralytics 8.3.145 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 50.8/112.6 GB disk)


In [ ]:
from ultralytics import YOLO

data_yaml = '/content/drive/MyDrive/Colab_Notebooks/dane/datasets/processed_data/data.yaml'

model = YOLO('yolov8n.pt')

# Trening
model.train(data=data_yaml, epochs=30, imgsz=640, device="cuda")

Ultralytics 8.3.145 🚀 Python-3.11.12 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/datasets/processed_data/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, 

train: Scanning /kaggle/datasets/processed_data/labels/train.cache... 136 images, 0 backgrounds, 0 corrupt: 100%|██████████| 136/136 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2096.7±342.3 MB/s, size: 1899.3 KB)



val: Scanning /kaggle/datasets/processed_data/labels/val.cache... 59 images, 0 backgrounds, 0 corrupt: 100%|██████████| 59/59 [00:00<?, ?it/s]


Plotting labels to runs/detect/train2/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs/detect/train2
Starting training for 30 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

                   all         59         59          1      0.983      0.995      0.765



30 epochs completed in 1.110 hours.
Optimizer stripped from runs/detect/train2/weights/last.pt, 6.2MB
Optimizer stripped from runs/detect/train2/weights/best.pt, 6.2MB

Validating runs/detect/train2/weights/best.pt...
Ultralytics 8.3.145 🚀 Python-3.11.12 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:14<00:00,  7.08s/it]


                   all         59         59          1      0.983      0.995      0.765
Speed: 1.7ms preprocess, 172.7ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to runs/detect/train2


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f55a27031d0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0, ...,           1]), array([[          1, ...,           0]]), 'Recall', 'Precision'], [array([          0, ...,           1]), array([[  0.0079617, ...,           0]]), 'Confidence', 'F1'], [array([          0, ...,           1]), array([[  0.0039967, ...,           1]]), 'Confidence', 'Precision'], [array([          0, ...,           1]), array([[          1, ...,           0]]), 'Confidence', 'Recall']]
fitness: 0.7877175076621864
keys: ['metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']
maps: array([     0.7647])
names: {0: 'plate'}
plot: True
results_dict: {'metrics/precision(B)': 1.0, 'met

**Wykrywanie**

In [25]:
import xml.etree.ElementTree as ET
from ultralytics import YOLO
import os

def load_annotations(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    annotations = {}
    for image in root.findall('image'):
        name = image.get('name')
        width = int(image.get('width'))
        height = int(image.get('height'))
        for box in image.findall('box'):
            if box.get('label') == 'plate':
                xtl = float(box.get('xtl'))
                ytl = float(box.get('ytl'))
                xbr = float(box.get('xbr'))
                ybr = float(box.get('ybr'))
                annotations[name] = (xtl, ytl, xbr, ybr, width, height)
    return annotations

def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    iou = interArea / float(boxAArea + boxBArea - interArea + 1e-6)
    return iou

annotations = load_annotations('/content/drive/MyDrive/Colab_Notebooks/dane/poland-vehicle-license-plate-dataset/annotations.xml')

model = YOLO("/content/drive/MyDrive/Colab_Notebooks/dane/runs/detect/train2/weights/best.pt")
model.to('cpu')
results = model.predict(source="/content/drive/MyDrive/Colab_Notebooks/dane/datasets/processed_data/images/val", save=True, conf=0.017, save_txt=True)

detected = 0
undetected = 0
ious = []

with open('/content/drive/MyDrive/Colab_Notebooks/wynik.txt', 'w') as f:
    for r in results:
        image_name = os.path.basename(r.path)
        gt = annotations.get(image_name)
        if len(r.boxes) > 0 and gt:
            detected += 1
            pred = r.boxes.xyxy[0].cpu().numpy()
            width, height = gt[4], gt[5]
            pred_box = [
                pred[0], pred[1], pred[2], pred[3]
            ]
            gt_box = [gt[0], gt[1], gt[2], gt[3]]
            iou = compute_iou(pred_box, gt_box)
            ious.append(iou)
            f.write(f"{image_name}: IoU={iou:.4f}\n")
        else:
            undetected += 1
            f.write(f"{image_name}: brak wykrycia\n")

    if ious:
        mean_iou = sum(ious) / len(ious)
    else:
        mean_iou = 0.0
    f.write(f"\nŚrednie IoU: {mean_iou:.4f}\n")
    f.write(f"Wykryto obiekty na {detected} obrazach.\n")
    f.write(f"Brak wykryć na {undetected} obrazach.\n")



image 1/59 /content/drive/MyDrive/Colab_Notebooks/dane/datasets/processed_data/images/val/10.jpg: 288x640 2 plates, 63.0ms
image 2/59 /content/drive/MyDrive/Colab_Notebooks/dane/datasets/processed_data/images/val/109.jpg: 448x640 1 plate, 88.1ms
image 3/59 /content/drive/MyDrive/Colab_Notebooks/dane/datasets/processed_data/images/val/110.jpg: 448x640 1 plate, 121.3ms
image 4/59 /content/drive/MyDrive/Colab_Notebooks/dane/datasets/processed_data/images/val/111.jpg: 448x640 1 plate, 124.0ms
image 5/59 /content/drive/MyDrive/Colab_Notebooks/dane/datasets/processed_data/images/val/112.jpg: 448x640 1 plate, 103.7ms
image 6/59 /content/drive/MyDrive/Colab_Notebooks/dane/datasets/processed_data/images/val/115.jpg: 448x640 (no detections), 105.7ms
image 7/59 /content/drive/MyDrive/Colab_Notebooks/dane/datasets/processed_data/images/val/122.jpg: 448x640 1 plate, 88.2ms
image 8/59 /content/drive/MyDrive/Colab_Notebooks/dane/datasets/processed_data/images/val/124.jpg: 448x640 1 plate, 117.9ms
im

**Wycinanie**

In [8]:
import cv2
import os
import random
import xml.etree.ElementTree as ET
from ultralytics import YOLO

photos_path = "/content/drive/MyDrive/Colab_Notebooks/dane/poland-vehicle-license-plate-dataset/photos"
all_images = [f for f in os.listdir(photos_path) if f.endswith('.jpg')]
random_images = random.sample(all_images, 100)
random_image_paths = [os.path.join(photos_path, img) for img in random_images]

annotations_path = "/content/drive/MyDrive/Colab_Notebooks/dane/poland-vehicle-license-plate-dataset/annotations.xml"
def load_annotations(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    annotations = {}
    for image in root.findall('image'):
        image_name = image.get('name')
        for box in image.findall('box'):
            plate_number = box.find("attribute[@name='plate number']").text
            annotations[image_name] = plate_number
    return annotations

annotations = load_annotations(annotations_path)

model = YOLO("/content/drive/MyDrive/Colab_Notebooks/dane/runs/detect/train2/weights/best.pt")
model.to('cpu')

def detect_license_plates(image_path, margin=1):
    image = cv2.imread(image_path)
    height, width, _ = image.shape

    results = model.predict(image, conf=0.02)
    detections = results[0].boxes

    if detections is None or detections.xyxy is None:
        return 0, []

    cropped_plate_paths = []
    os.makedirs("/content/drive/MyDrive/Colab_Notebooks/dane/detected_plates", exist_ok=True)

    for i, box in enumerate(detections.xyxy):
        x1, y1, x2, y2 = map(int, box[:4])

        w_margin = int((x2 - x1) * margin)
        h_margin = int((y2 - y1) * margin)
        x1_new = max(0, x1 - w_margin)
        y1_new = max(0, y1 - h_margin)
        x2_new = min(width, x2 + w_margin)
        y2_new = min(height, y2 + h_margin)

        cropped_plate = image[y1_new:y2_new, x1_new:x2_new]
        output_path = f"/content/drive/MyDrive/Colab_Notebooks/dane/detected_plates/plate_{os.path.basename(image_path).split('.')[0]}_{i}.jpg"
        cv2.imwrite(output_path, cropped_plate)
        cropped_plate_paths.append(output_path)

    return len(detections), cropped_plate_paths

for i, img_path in enumerate(random_image_paths):
    count, paths = detect_license_plates(img_path)
    print(f"[{i+1}/100] Detected {count} plates in {os.path.basename(img_path)}")
    for path in paths:
        print("Zapisano:", path)


0: 448x640 1 plate, 168.8ms
Speed: 6.1ms preprocess, 168.8ms inference, 1.6ms postprocess per image at shape (1, 3, 448, 640)
[1/100] Detected 1 plates in 131.jpg
  --> Saved: /content/drive/MyDrive/Colab_Notebooks/dane/detected_plates/plate_131_0.jpg

0: 448x640 (no detections), 104.7ms
Speed: 4.0ms preprocess, 104.7ms inference, 0.6ms postprocess per image at shape (1, 3, 448, 640)
[2/100] Detected 0 plates in 115.jpg

0: 448x640 1 plate, 153.6ms
Speed: 4.4ms preprocess, 153.6ms inference, 1.0ms postprocess per image at shape (1, 3, 448, 640)
[3/100] Detected 1 plates in 133.jpg
  --> Saved: /content/drive/MyDrive/Colab_Notebooks/dane/detected_plates/plate_133_0.jpg

0: 448x640 1 plate, 116.6ms
Speed: 3.7ms preprocess, 116.6ms inference, 1.2ms postprocess per image at shape (1, 3, 448, 640)
[4/100] Detected 1 plates in 190.jpg
  --> Saved: /content/drive/MyDrive/Colab_Notebooks/dane/detected_plates/plate_190_0.jpg

0: 448x640 1 plate, 100.7ms
Speed: 3.9ms preprocess, 100.7ms inferen

**Paddleocr**

In [ ]:
!python -m pip install paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu118/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 GB 667.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 16.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 17.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 699.9/699.9 MB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 MB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install paddleocr==3.0.0

In [ ]:
from paddleocr import PaddleOCR

In [ ]:
ocr=PaddleOCR(lang='en', use_doc_orientation_classify=False, use_doc_unwarping=False)

Creating model: ('PP-LCNet_x0_25_textline_ori', None)
Using official model (PP-LCNet_x0_25_textline_ori), the model files will be automatically downloaded and saved in /root/.paddlex/official_models.


Connecting to https://paddle-model-ecology.bj.bcebos.com/paddlex/official_inference_model/paddle3.0.0/PP-LCNet_x0_25_textline_ori_infer.tar ...
[==================================================] 100.00%
Extracting PP-LCNet_x0_25_textline_ori_infer.tar
[==================================================] 100.00%


/usr/local/lib/python3.11/dist-packages/paddle/utils/cpp_extension/extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_mobile_det', None)
Using official model (PP-OCRv5_mobile_det), the model files will be automatically downloaded and saved in /root/.paddlex/official_models.


Connecting to https://paddle-model-ecology.bj.bcebos.com/paddlex/official_inference_model/paddle3.0.0/PP-OCRv5_mobile_det_infer.tar ...
[==================================================] 100.00%
Extracting PP-OCRv5_mobile_det_infer.tar
[==================================================] 100.00%


Creating model: ('PP-OCRv5_mobile_rec', None)
Using official model (PP-OCRv5_mobile_rec), the model files will be automatically downloaded and saved in /root/.paddlex/official_models.


Connecting to https://paddle-model-ecology.bj.bcebos.com/paddlex/official_inference_model/paddle3.0.0/PP-OCRv5_mobile_rec_infer.tar ...
[==================================================] 100.00%
Extracting PP-OCRv5_mobile_rec_infer.tar
[==================================================] 100.00%


In [26]:
import os
import time
from tqdm import tqdm

# Zbieramy wyniki
all_results = []

plates_folder = '/content/drive/MyDrive/Colab_Notebooks/dane/detected_plates'
plate_images = [f for f in os.listdir(plates_folder) if f.endswith('.jpg')]

start_time = time.time()

for img_name in tqdm(plate_images):
    img_path = os.path.join(plates_folder, img_name)
    result = ocr.predict(img_path)
    for item in result:
        all_results.append((img_path, item['rec_texts']))

end_time = time.time()

elapsed_time = end_time - start_time

with open("/content/drive/MyDrive/Colab_Notebooks/wynik.txt", "a") as f:
    f.write(f"Czas odczytu: {elapsed_time:.2f} sekund\n")


100%|██████████| 105/105 [00:15<00:00,  6.92it/s]


In [27]:
import os
import re
import xml.etree.ElementTree as ET

def extract_plate_text(ocr_texts):
    for text in ocr_texts:
        cleaned = re.sub(r'[^A-Z0-9]', '', text.upper())
        if (re.search(r'[A-Z]', cleaned) and
            re.search(r'\d', cleaned) and
            len(cleaned) > 3):
            return cleaned
    return None

def get_image_key_from_path(path):
    match = re.search(r'plate_(\d+)_\d+\.jpg$', path)
    if match:
        return match.group(1)
    return None

def load_annotations_from_xml(xml_path):
    annotations = {}
    tree = ET.parse(xml_path)
    root = tree.getroot()

    for image in root.iter('image'):
        img_name = image.attrib['name']
        plate_number = None
        for box in image.findall('box'):
            for attr in box.findall('attribute'):
                if attr.attrib['name'] == 'plate number':
                    plate_number = attr.text.strip().upper()
                    break
            if plate_number:
                break

        if plate_number:
            base_name = os.path.splitext(img_name)[0]
            annotations[base_name] = plate_number
    return annotations

xml_folder = '/content/drive/MyDrive/Colab_Notebooks/dane/poland-vehicle-license-plate-dataset/annotations.xml'
annotations = load_annotations_from_xml(xml_folder)

filtered_results = []
for path, texts in all_results:
    plate = extract_plate_text(texts)
    if plate:
        filtered_results.append((path, plate))

correct = 0
total = 0

for path, ocr_plate in filtered_results:
    key = get_image_key_from_path(path)
    if key and key in annotations:
        true_plate = annotations[key]
        total += 1
        if ocr_plate == true_plate:
            correct += 1
        print(f"{path} → OCR: {ocr_plate}, XML: {true_plate}, MATCH: {ocr_plate == true_plate}")

if total:
    accuracy = correct / total
    print(f"\nDokładność: {correct}/{total} = {accuracy:.2%}")
else:
    print("Brak dopasowanych rekordów")

with open("/content/drive/MyDrive/Colab_Notebooks/wynik.txt", "a") as f:
    if total:
        f.write(f"Dokładność: {correct}/{total} = {accuracy:.2%}\n")


/content/drive/MyDrive/Colab_Notebooks/dane/detected_plates/plate_131_0.jpg → OCR: SL98181, XML: SL98181, MATCH: True
/content/drive/MyDrive/Colab_Notebooks/dane/detected_plates/plate_133_0.jpg → OCR: SW60961, XML: SW60961, MATCH: True
/content/drive/MyDrive/Colab_Notebooks/dane/detected_plates/plate_190_0.jpg → OCR: SP116111, XML: SPI16111, MATCH: False
/content/drive/MyDrive/Colab_Notebooks/dane/detected_plates/plate_173_0.jpg → OCR: SK959YF, XML: SK959YF, MATCH: True
/content/drive/MyDrive/Colab_Notebooks/dane/detected_plates/plate_124_0.jpg → OCR: SW55815, XML: SW55815, MATCH: True
/content/drive/MyDrive/Colab_Notebooks/dane/detected_plates/plate_99_0.jpg → OCR: CB441ML, XML: CB441ML, MATCH: True
/content/drive/MyDrive/Colab_Notebooks/dane/detected_plates/plate_19_0.jpg → OCR: SJ3132A, XML: SJ3132A, MATCH: True
/content/drive/MyDrive/Colab_Notebooks/dane/detected_plates/plate_32_0.jpg → OCR: SMI03L9, XML: SMI03L9, MATCH: True
/content/drive/MyDrive/Colab_Notebooks/dane/detected_pla

**Ocena**

In [28]:
def calculate_final_grade(accuracy_percent: float, processing_time_sec: float) -> float:
    if accuracy_percent < 60 or processing_time_sec > 60:
        return 2.0

    accuracy_norm = (accuracy_percent - 60) / 40

    time_norm = (60 - processing_time_sec) / 50

    score = 0.7 * accuracy_norm + 0.3 * time_norm

    grade = 2.0 + 3.0 * score

    return round(grade * 2) / 2

def parse_results_file(file_path: str):
    processing_time = None
    accuracy_percent = None

    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith("Czas odczytu:"):
                parts = line.split()
                try:
                    processing_time = float(parts[2])
                except:
                    pass
            elif line.startswith("Dokładność:"):
                match = re.search(r"=\s*([\d\.]+)%", line)
                if match:
                    accuracy_percent = float(match.group(1))

    return processing_time, accuracy_percent


file_path = "/content/drive/MyDrive/Colab_Notebooks/wynik.txt"

processing_time, accuracy_percent = parse_results_file(file_path)

if processing_time is not None and accuracy_percent is not None:
    grade = calculate_final_grade(accuracy_percent, processing_time)
    print(f"Ocena końcowa: {grade}")

    with open(file_path, "a") as f:
        f.write(f"Ocena końcowa: {grade}\n")
else:
    print("Nie udało się odczytać danych do oceny końcowej.")


Ocena końcowa: 4.0


In [ ]:
!git